In [1]:
import pandas as pd
from datetime import datetime
import os

hours_log = pd.read_csv("hours.csv")
hours_log["Date"] = pd.to_datetime(hours_log["Date"], format="%d.%m.%Y")
hours_log["Start"] = pd.to_datetime(hours_log["Start"], format="%H:%M")
hours_log["End"] = pd.to_datetime(hours_log["End"], format="%H:%M")
hours_log["Hours"] = (hours_log["End"] - hours_log["Start"]).dt.total_seconds() / 3600

# Clean up tags formatting 
hours_log["Tag"] = hours_log["Tag"].str.strip()

# Summarize hours by contributor
summary = hours_log.groupby("Name")["Hours"].sum().reset_index()
summary = summary.sort_values(by="Hours", ascending=False)

# Summarize hours by tag
tag_summary = hours_log.groupby("Tag")["Hours"].sum().reset_index()
tag_summary = tag_summary.sort_values(by="Hours", ascending=False)

# Projected final hours calculation (updated dates to 2026 Spring semester)
project_start = datetime(2026, 3, 25)
project_end = datetime(2026, 4, 20)
last_log_day = hours_log["Date"].max()

days_passed = (last_log_day - project_start).days + 1  # include start day
total_days = (project_end - project_start).days + 1

# Estimate final hours per author
summary["Hours per day"] = summary["Hours"] / days_passed
summary["Estimated final hours"] = summary["Hours per day"] * total_days

print(f"{'=' * 10} Development Hours Report {'=' * 10}")
display("# Total Hours by Contributor:", summary)
display("# Total Hours by Tag:", tag_summary)

========== Development Hours Report ==========


'# Total Hours by Contributor:'

,Name,Hours,Hours per day,Estimated final hours
3,vaakir,33.333333,2.222222,60.00
2,togun,21.716667,1.447778,39.09
1,GabIsr,11.666667,0.777778,21.00
0,BaalCat,9.500000,0.633333,17.10


'# Total Hours by Tag:'

,Tag,Hours
6,System 1,33.833333
2,Planning,12.550000
1,Implementation,10.833333
4,Reading Theory & Coding,7.750000
5,Refactor,4.000000
0,Fix,3.000000
3,Reading & Exploration,2.750000
7,System 2,1.500000


In [ ]:
from pathlib import Path

EXCLUDE_DIRS = {"node_modules", "__pycache__", "data", ".pytest_cache", ".conda"}
EXCLUDE_FILES = {
    "_momemtum.py",
    "_others.py",
    "_trend.py",
    "_volatility.py",
    "_volume.py",
}
CODE_EXTENSIONS = {".py", ".ts", ".tsx", ".js", ".jsx", ".html", ".css"}


def count_lines(root: Path):
    total = 0
    file_count = 0

    for path in root.iterdir():
        if path.is_dir():
            if path.name in EXCLUDE_DIRS:
                continue
            sub_total, sub_files = count_lines(path)
            total += sub_total
            file_count += sub_files

        else:
            if path.name in EXCLUDE_FILES:
                continue

            if path.suffix in CODE_EXTENSIONS:
                try:
                    with path.open("r", encoding="utf-8", errors="ignore") as f:
                        lines = sum(1 for _ in f)
                        total += lines
                        # print(lines, path) # debug where what kinda illigal file was included
                        file_count += 1
                except OSError:
                    pass

    return total, file_count


if __name__ == "__main__":
    root = Path("../").resolve()
    total_lines, total_files = count_lines(root)
    print(f"Total lines: {total_lines}, Total files: {total_files}")